In [2]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
np.random.seed(42)

In [3]:
TEST_SIZE = 0.1 # 10% of non-excluded data

In [4]:
BASE_DIR = "../"
if not os.path.isdir("merged"):
    os.makedirs("merged")

In [5]:
human_dfs = []
ai_dfs = []

for f in Path("data").iterdir():
    df = pd.read_csv(f)
    df["file"] = f.name[:-4] # file name only
    if f.name.startswith("ai_"):
        ai_dfs.append(df)
    else:
        human_dfs.append(df)

human_df = pd.concat(human_dfs)
ai_df = pd.concat(ai_dfs)

In [6]:
# make testing data similar in weight
def balance(testing_df, label):
    df_0 = testing_df[testing_df[label] == 0]
    df_1 = testing_df[testing_df[label] == 1]
    sample_size = min(len(df_0), len(df_1))

    df_0 = df_0.sample(n=sample_size, random_state=42)
    df_1 = df_1.sample(n=sample_size, random_state=42)
    testing_df = pd.concat([df_0, df_1]).sample(frac=1, random_state=42)
    testing_df = testing_df.reset_index(drop=True)
    return testing_df

In [7]:
for f in Path("data").iterdir():
    # exclude type f
    file_name = f.name[:-4]
    if file_name.startswith("ai"):
        continue
    training_mask = np.where(human_df["file"] == file_name, False, np.random.rand(len(human_df)) > TEST_SIZE)
    training_df = human_df[training_mask][["text", "label0", "label1"]]
    testing_df = human_df[~training_mask][["text", "label0", "label1"]]
    print(f"[{file_name}_as_test] Training: {len(training_df)}, Testing: {len(testing_df)}")
    csv_path = f"merged/{file_name}_as_test"
    if not os.path.isdir(csv_path):
        os.makedirs(csv_path)

    training_df.to_csv(f"{csv_path}/training.csv", index=False)
    testing_df = balance(testing_df, "label1")
    testing_df.to_csv(f"{csv_path}/testing.csv", index=False)

[onion_as_test] Training: 6198, Testing: 1725
[arxiv_summaries_as_test] Training: 6220, Testing: 1703
[bnc_as_test] Training: 6311, Testing: 1612
[nature_as_test] Training: 7102, Testing: 821
[speech_as_test] Training: 6865, Testing: 1058
[political_manifestos_as_test] Training: 7026, Testing: 897
[arxiv_abstracts_as_test] Training: 6239, Testing: 1684
[fake_news_as_test] Training: 5581, Testing: 2342
[wiki_as_test] Training: 6287, Testing: 1636
[spam_messages_as_test] Training: 6499, Testing: 1424
[mission_as_test] Training: 7034, Testing: 889
[vision_mission_as_test] Training: 7013, Testing: 910


In [8]:
training_mask = np.random.rand(len(human_df)) > TEST_SIZE
training_all_df = human_df[training_mask][["text", "label0", "label1"]].reset_index(drop=True)
testing_all_df = human_df[~training_mask][["text", "label0", "label1"]].reset_index(drop=True)
testing_all_df = balance(testing_all_df, "label1")

training_all_df.to_csv("merged/bullshit_training_all.csv", index=False)
testing_all_df.to_csv("merged/bullshit_testing_all.csv", index=False)

print(f"training_all: {len(training_all_df)}, testing_all: {len(testing_all_df)}")

training_all: 7110, testing_all: 636


In [10]:
# Generate training data for human vs AI

human_df.info()
ai_df.info()

all_df = pd.concat([human_df, ai_df])
training_mask = np.random.rand(len(all_df)) > TEST_SIZE
training_all_df = all_df[training_mask][["text", "label0", "label1"]].reset_index(drop=True)
testing_all_df = all_df[~training_mask][["text", "label0", "label1"]].reset_index(drop=True)
testing_all_df = balance(testing_all_df, "label0")

training_all_df.to_csv("merged/ai_training_all.csv", index=False)
testing_all_df.to_csv("merged/ai_testing_all.csv", index=False)

print(f"training_all: {len(training_all_df)}, testing_all: {len(testing_all_df)}")

<class 'pandas.DataFrame'>
Index: 7923 entries, 0 to 73
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    7923 non-null   str  
 1   label0  7923 non-null   int64
 2   label1  7923 non-null   int64
 3   file    7923 non-null   str  
dtypes: int64(2), str(2)
memory usage: 83.2 MB
<class 'pandas.DataFrame'>
Index: 1660 entries, 0 to 1241
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    1660 non-null   str  
 1   label0  1660 non-null   int64
 2   label1  1660 non-null   int64
 3   file    1660 non-null   str  
dtypes: int64(2), str(2)
memory usage: 11.3 MB
training_all: 8596, testing_all: 334
